In [1]:
import pandas as pd
import unicodedata
import unidecode



In [2]:
# Helper function to normalize strings (remove diacritics, convert to lowercase, and strip whitespace)
def normalize_str(x):
    if pd.isnull(x):
        return ""
    return unidecode.unidecode(x).lower().strip()



In [ ]:
# File paths 
invites_csv_path = "data/masters-com-data/2025-invites.csv"
players_json_path = "data/pgatour-com-data/players.json"

# Read the invites CSV file and the players JSON file
invites_df = pd.read_csv(invites_csv_path)
players_df = pd.read_json(players_json_path)

# Create normalized name columns in invites_df (assuming columns 'firstname' and 'lastname')
invites_df["norm_firstname"] = invites_df["firstname"].apply(normalize_str)
invites_df["norm_lastname"] = invites_df["lastname"].apply(normalize_str)
invites_df["norm_name"] = invites_df["norm_firstname"] + " " + invites_df["norm_lastname"]

# Create normalized name columns in players_df (assuming keys 'firstName' and 'lastName')
players_df["norm_firstname"] = players_df["firstName"].apply(normalize_str)
players_df["norm_lastname"] = players_df["lastName"].apply(normalize_str)
players_df["norm_name"] = players_df["norm_firstname"] + " " + players_df["norm_lastname"]

# Merge invites_df with players_df on the normalized name column.
# The 'id' field from players_df is the PGATOUR_ID.
merged_df = invites_df.merge(
    players_df[["norm_name", "id"]],
    on="norm_name",
    how="left"
)

# Rename 'id' to 'PGATOUR_ID'
merged_df.rename(columns={"id": "PGATOUR_ID"}, inplace=True)

# Drop helper normalization columns if no longer needed
merged_df.drop(columns=["norm_firstname", "norm_lastname", "norm_name"], inplace=True)

# Identify players missing PGATOUR_ID (i.e. invitees not found in players_df)
missing_df = merged_df[merged_df["PGATOUR_ID"].isna()]

# Create a list of names from the missing entries (combining firstname and lastname)
missing_players = missing_df.apply(lambda row: f"{row['firstname']} {row['lastname']}", axis=1).tolist()

print("Players missing from the players dataset:")
for name in missing_players:
    print(name)

Players missing from the players dataset:
Jose Luis  Ballester
Evan Beck
Cameron Davis
Nicolas Echavarria
Noah Kent
Hiroshi Tai


In [6]:
merged_df.to_csv("invites_csv_path", mode='w', header=True, index=False)